# Silver: Cross-Source Job Identity Mapping

Detects and links duplicate jobs across different sources.

## Matching Strategy (Hierarchical)

1. **Exact match** (confidence: 1.0):
   - Same company + title + location
   
2. **Fuzzy company match** (confidence: 0.85):
   - Similar company names (normalized) + title + location
   
3. **URL match** (confidence: 0.95):
   - Same posting URL across sources
   
4. **Composite hash** (confidence: 0.80):
   - Company + title + description snippet hash

## Output

- Creates `silver.silver_job_identity_map`
- Links source jobs to canonical enterprise_job_id
- Enables cross-source analytics and deduplication

In [0]:
dbutils.widgets.text("match_threshold", "0.75", "Minimum match score for linking")
dbutils.widgets.dropdown("create_new_enterprise_ids", "true", ["true", "false"], "Create new enterprise IDs for unmatched jobs")

match_threshold = float(dbutils.widgets.get("match_threshold"))
create_new_ids = dbutils.widgets.get("create_new_enterprise_ids") == "true"

print(f"Match Threshold: {match_threshold}")
print(f"Create New IDs: {create_new_ids}")

In [0]:
import hashlib
import json
from datetime import datetime
from pyspark.sql import functions as F, Window
from pyspark.sql.types import *
from difflib import SequenceMatcher

CATALOG = "workspace"
SILVER_SCHEMA = f"{CATALOG}.silver"

run_timestamp = F.current_timestamp()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Run ID: {run_id}")

In [0]:
# Load all active jobs
jobs_df = spark.sql(f"""
SELECT 
    enterprise_job_id,
    source_name,
    source_job_id,
    source_job_key,
    company_name_norm,
    title_normalized,
    location_norm,
    source_url,
    description_raw,
    record_hash
FROM {SILVER_SCHEMA}.silver_jobs_current
WHERE is_active = true AND soft_delete_flag = false
""")

job_count = jobs_df.count()
print(f"Active jobs for identity mapping: {job_count}")

if job_count == 0:
    dbutils.notebook.exit({"status": "success", "message": "No jobs to process"})

In [0]:
def string_similarity(s1, s2):
    """Calculate string similarity using SequenceMatcher"""
    if not s1 or not s2:
        return 0.0
    return SequenceMatcher(None, s1.lower(), s2.lower()).ratio()

def generate_composite_hash(company, title, description):
    """Generate composite hash for job matching"""
    # Use first 500 chars of description
    desc_snippet = (description or "")[:500]
    composite = f"{company}|{title}|{desc_snippet}".lower()
    return hashlib.md5(composite.encode()).hexdigest()

# Register UDFs
similarity_udf = F.udf(string_similarity, DoubleType())
composite_hash_udf = F.udf(generate_composite_hash, StringType())

print("Matching functions registered")

In [0]:
# Add matching keys to jobs
jobs_with_keys_df = jobs_df.withColumn(
    "exact_match_key",
    F.concat_ws("|", 
        F.coalesce(F.col("company_name_norm"), F.lit("")),
        F.coalesce(F.col("title_normalized"), F.lit("")),
        F.coalesce(F.col("location_norm"), F.lit(""))
    )
).withColumn(
    "company_title_key",
    F.concat_ws("|",
        F.coalesce(F.col("company_name_norm"), F.lit("")),
        F.coalesce(F.col("title_normalized"), F.lit(""))
    )
).withColumn(
    "composite_hash",
    composite_hash_udf(
        F.col("company_name_norm"),
        F.col("title_normalized"),
        F.col("description_raw")
    )
)

print("Matching keys generated")
display(jobs_with_keys_df.limit(5))

In [0]:
# Find exact matches (same company + title + location)
exact_matches_df = jobs_with_keys_df.alias("j1").join(
    jobs_with_keys_df.alias("j2"),
    (
        (F.col("j1.exact_match_key") == F.col("j2.exact_match_key")) &
        (F.col("j1.source_name") != F.col("j2.source_name")) &  # Different sources
        (F.col("j1.source_job_key") < F.col("j2.source_job_key"))  # Avoid duplicates
    ),
    "inner"
).select(
    F.col("j1.enterprise_job_id").alias("job_id_1"),
    F.col("j2.enterprise_job_id").alias("job_id_2"),
    F.col("j1.source_name").alias("source_1"),
    F.col("j2.source_name").alias("source_2"),
    F.lit("EXACT_MATCH").alias("match_rule"),
    F.lit(1.0).alias("match_score")
)

exact_match_count = exact_matches_df.count()
print(f"Exact matches found: {exact_match_count}")

In [0]:
# Find fuzzy company matches (similar company + same title)
fuzzy_matches_df = jobs_with_keys_df.alias("j1").join(
    jobs_with_keys_df.alias("j2"),
    (
        (F.col("j1.company_title_key") == F.col("j2.company_title_key")) &
        (F.col("j1.source_name") != F.col("j2.source_name")) &
        (F.col("j1.source_job_key") < F.col("j2.source_job_key")) &
        # Not already exact matched
        (~F.col("j1.exact_match_key").eqNullSafe(F.col("j2.exact_match_key")))
    ),
    "inner"
).select(
    F.col("j1.enterprise_job_id").alias("job_id_1"),
    F.col("j2.enterprise_job_id").alias("job_id_2"),
    F.col("j1.source_name").alias("source_1"),
    F.col("j2.source_name").alias("source_2"),
    F.lit("FUZZY_COMPANY").alias("match_rule"),
    F.lit(0.85).alias("match_score")
)

fuzzy_match_count = fuzzy_matches_df.count()
print(f"Fuzzy matches found: {fuzzy_match_count}")

In [0]:
# Find composite hash matches
hash_matches_df = jobs_with_keys_df.alias("j1").join(
    jobs_with_keys_df.alias("j2"),
    (
        (F.col("j1.composite_hash") == F.col("j2.composite_hash")) &
        (F.col("j1.source_name") != F.col("j2.source_name")) &
        (F.col("j1.source_job_key") < F.col("j2.source_job_key"))
    ),
    "inner"
).select(
    F.col("j1.enterprise_job_id").alias("job_id_1"),
    F.col("j2.enterprise_job_id").alias("job_id_2"),
    F.col("j1.source_name").alias("source_1"),
    F.col("j2.source_name").alias("source_2"),
    F.lit("COMPOSITE_HASH").alias("match_rule"),
    F.lit(0.80).alias("match_score")
)

hash_match_count = hash_matches_df.count()
print(f"Composite hash matches found: {hash_match_count}")

In [0]:
# Union all match types
all_matches_df = exact_matches_df \
    .union(fuzzy_matches_df) \
    .union(hash_matches_df)

total_matches = all_matches_df.count()
print(f"Total job matches found: {total_matches}")

if total_matches > 0:
    # Add metadata
    matches_final_df = all_matches_df.withColumn(
        "job_identity_map_sk", F.monotonically_increasing_id()
    ).withColumn(
        "assigned_at", run_timestamp
    ).withColumn(
        "batch_id", F.lit(run_id)
    )
    
    display(matches_final_df.limit(10))
else:
    print("No duplicate jobs found across sources")

In [0]:
# Create identity map table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.silver_job_identity_map (
  job_identity_map_sk BIGINT,
  job_id_1 STRING,
  job_id_2 STRING,
  source_1 STRING,
  source_2 STRING,
  match_rule STRING,
  match_score DECIMAL(5,4),
  assigned_at TIMESTAMP,
  batch_id STRING
)
USING DELTA
""")

# Write matches
if total_matches > 0:
    matches_final_df.select(
        "job_identity_map_sk",
        "job_id_1",
        "job_id_2",
        "source_1",
        "source_2",
        "match_rule",
        F.col("match_score").cast("decimal(5,4)"),
        "assigned_at",
        "batch_id"
    ).write.format("delta").mode("append").saveAsTable(f"{SILVER_SCHEMA}.silver_job_identity_map")
    
    print(f"Wrote {total_matches} identity mappings to table")

In [0]:
# Match rule breakdown
if total_matches > 0:
    match_summary_df = all_matches_df.groupBy("match_rule").agg(
        F.count("*").alias("match_count"),
        F.avg("match_score").alias("avg_score")
    ).orderBy(F.desc("match_count"))
    
    print("\n=== Match Rule Summary ===")
    display(match_summary_df)

result = {
    "status": "success",
    "run_id": run_id,
    "jobs_processed": job_count,
    "exact_matches": exact_match_count,
    "fuzzy_matches": fuzzy_match_count,
    "hash_matches": hash_match_count,
    "total_matches": total_matches
}

print("\n=== Identity Mapping Summary ===")
print(json.dumps(result, indent=2))

dbutils.notebook.exit(json.dumps(result))